In [1]:
import pandas as pd

import re
import string

In [2]:
# split by tabs not commas
df = pd.read_csv('SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])

# Step 2: EDA

## Class imbalance

* What's the ratio of ham to spam?
* What's the level of imbalance? Is it critical?
* What are we going to do about this in the future?

In [3]:
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [4]:
df['label'].value_counts(normalize=True)

label
ham     0.865937
spam    0.134063
Name: proportion, dtype: float64

With 86.6% ham / 13.4% spam:

* We'll **not use SMOTE** (that's for more severe imbalance, and it can create synthetic text that doesn't make grammatical sense — SMOTE was designed for numeric/tabular features, not raw text).
* Instead we'll use **LogisticRegression(class_weight='balanced')**

## Are spam messages longer or shorter than ham?

In [5]:
# add length column
df['length'] = df['message'].apply(len)

In [6]:
df.groupby('label')['length'].mean()

label
ham      71.482487
spam    138.670683
Name: length, dtype: float64

In [7]:
df.groupby('label')['length'].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
ham,4825.0,71.482487,58.440652,2.0,33.0,52.0,93.0,910.0
spam,747.0,138.670683,28.873603,13.0,133.0,149.0,157.0,223.0


That 71-vs-139-character gap means length itself can be a **useful input feature**, separate from the words.

The spam message needs to contain to work: a hook, an offer, urgency, and usually a call-to-action (a phone number, a link, "reply STOP to unsubscribe" — required by SMS regulations).

In [8]:
# how much duplicates we have
df.duplicated().sum()

np.int64(403)

In [9]:
df[df.duplicated()].head()

,label,message,length
103,ham,As per your request 'Melle Melle (Oru Minnamin...,160
154,ham,As per your request 'Melle Melle (Oru Minnamin...,160
207,ham,"As I entered my cabin my PA said, '' Happy B'd...",156
223,ham,"Sorry, I'll call later",22
326,ham,No calls..messages..missed calls,32


Droping duplicates before splitting into train/test because we need the model correctly classify a message it has never seen before not recognizing concrete string

In [10]:
print("df.shape before removing duplicates:", df.shape)
df = df.drop_duplicates()
print("df.shape after removing duplicates:", df.shape)

df.shape before removing duplicates: (5572, 3)
df.shape after removing duplicates: (5169, 3)


# Step 3: Text preprocessing

1. **Lowercasing.** Without this, "FREE", "Free", and "free" are treated as three completely different words/tokens by any vectorizer. They mean the same thing, so we collapse them into one.

2. **Removing punctuation.** "prize!!!" and "prize" should count as the same word. Punctuation itself carries some signal (spam often has more ! and $), but we typically capture that separately as its own numeric feature, not by leaving punctuation glued to words.

3. **Removing stopwords.** Words like "the," "is," "at," "and" appear constantly in every message, spam or ham, and carry almost no information about which class a message belongs to. They add noise and bulk to your vocabulary without helping the model discriminate.

4. **Tokenization.** Splitting a sentence into individual words (tokens): "win free cash now" → ["win", "free", "cash", "now"]. This is the step that turns a string into a list the vectorizer can count.

5. **Stemming or lemmatization.** "winning," "wins," "won," "win" are different words to a computer but the same underlying idea to a human. Stemming crudely chops word endings (winning → win); lemmatization is smarter and uses actual grammar rules (better → good, is a lemma of "be"). Stemming is faster and simpler — reasonable for TF-IDF + LogReg on a small dataset like this.

As long as we extract signal like `num_exclamations`, `has_currency_symbol`, `pct_uppercase` as their own numeric columns before we lowercase/strip punctuation, we lose nothing — the raw text gets normalized for the vectorizer, but the "shoutiness" and symbols still exist as separate features the model can use.

In [11]:
# --- Pass 1: extract signal from RAW text, before cleaning destroys it ---
df['num_exclamations'] = df['message'].apply(lambda x: x.count('!'))
df['has_currency_symbol'] = df['message'].apply(lambda x: 1 if re.search(r'[$£€]', x) else 0)
df['pct_uppercase'] = df['message'].apply(  # raw counts are biased by length; proportions normalize for it
    lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0
)
# tells you "how many digit characters total" but can't distinguish where they are or what pattern they form
df['num_digits'] = df['message'].apply(lambda x: sum(1 for c in x if c.isdigit()))

# --- Pass 2: clean the text for the vectorizer ---
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # remove digits (already captured as num_digits)
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # collapse extra whitespace
    return text

df['clean_message'] = df['message'].apply(clean_text)

In [12]:
df

,label,message,length,num_exclamations,has_currency_symbol,pct_uppercase,num_digits,clean_message
0,ham,"Go until jurong point, crazy.. Available only ...",111,0,0,0.027027,0,go until jurong point crazy available only in ...
1,ham,Ok lar... Joking wif u oni...,29,0,0,0.068966,0,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,0,0,0.064516,25,free entry in a wkly comp to win fa cup final ...
3,ham,U dun say so early hor... U c already then say...,49,0,0,0.040816,0,u dun say so early hor u c already then say
4,ham,"Nah I don't think he goes to usf, he lives aro...",61,0,0,0.032787,0,nah i dont think he goes to usf he lives aroun...
...,...,...,...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,160,1,1,0.056250,21,this is the nd time we have tried contact u u ...
5568,ham,Will ü b going to esplanade fr home?,36,0,0,0.027778,0,will ü b going to esplanade fr home
5569,ham,"Pity, * was in mood for that. So...any other s...",57,0,0,0.035088,0,pity was in mood for that soany other suggestions
5570,ham,The guy did some bitching but I acted like i'd...,125,0,0,0.016000,0,the guy did some bitching but i acted like id ...


In [13]:
df.head()

,label,message,length,num_exclamations,has_currency_symbol,pct_uppercase,num_digits,clean_message
0,ham,"Go until jurong point, crazy.. Available only ...",111,0,0,0.027027,0,go until jurong point crazy available only in ...
1,ham,Ok lar... Joking wif u oni...,29,0,0,0.068966,0,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,0,0,0.064516,25,free entry in a wkly comp to win fa cup final ...
3,ham,U dun say so early hor... U c already then say...,49,0,0,0.040816,0,u dun say so early hor u c already then say
4,ham,"Nah I don't think he goes to usf, he lives aro...",61,0,0,0.032787,0,nah i dont think he goes to usf he lives aroun...


In [14]:
df[['message','clean_message']].sample(5)

,message,clean_message
3181,There the size of elephant tablets & u shove u...,there the size of elephant tablets u shove um ...
1049,I walked an hour 2 c u! doesnt that show I ca...,i walked an hour c u doesnt that show i care ...
1849,I dont want to hear philosophy. Just say what ...,i dont want to hear philosophy just say what h...
616,Happy valentines day I know its early but i ha...,happy valentines day i know its early but i ha...
2128,Tessy..pls do me a favor. Pls convey my birthd...,tessypls do me a favor pls convey my birthday ...
